In [1]:
%pip install "sagemaker<3" -q
!pip install pyathena awswrangler  --quiet
!pip install 'boto3>1.17.21' -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-mlops 1.10.1 requires sagemaker-core>=2.10.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.10.1 requires sagemaker-core>=2.10.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have 

Note: you may need to restart the kernel to use updated packages.


# Set up Athena Pneumonia DB
This notebook is purely for a user to register the pneumonia db in thier Glue Catalog

In [2]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect
import awswrangler as wr

sess = sagemaker.Session()
default_bucket = sess.default_bucket()
region = boto3.Session().region_name
bucket = "pneumonia-data-set-group-4"

# Athena staging directory (uses YOUR default bucket for query results)
s3_staging_dir = f"s3://{default_bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

print(f"Region: {region}")
print(f"Data bucket: {bucket}")
print(f"Staging dir: {s3_staging_dir}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region: us-east-1
Data bucket: pneumonia-data-set-group-4
Staging dir: s3://sagemaker-us-east-1-072753725776/athena/staging


In [3]:
database_name = "pneumonia_db"

### Drop the DB if it exists

In [4]:
drop_statement = f"""
DROP TABLE IF EXISTS {database_name}.image_metadata
"""
pd.read_sql(drop_statement, conn)

/tmp/ipykernel_813/2168075215.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(drop_statement, conn)


""


## Create table

In [5]:
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id              STRING,
    raw_s3_key            STRING,
    preprocessed_s3_key   STRING,
    label                 STRING,
    label_int             TINYINT,
    split                 STRING,
    source                STRING,
    file_type             STRING,
    pixel_mean            DOUBLE,
    pixel_std             DOUBLE,
    img_height            INT,
    img_width             INT,
    event_time            STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print("Table created!")

/tmp/ipykernel_813/1396329963.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


Table created!


### Validate the table was created and query it

In [6]:
import awswrangler as wr

In [8]:
query = f"SELECT * FROM {database_name}.image_metadata LIMIT 100"

df_meta = wr.athena.read_sql_query(
    query,
    database=database_name
)

In [15]:
try:
    if not df_meta.empty:
        print(f'Successfully pulled from {database_name}')
        
    else:
        print('FAILED - Pulled in data is blank, please re-try setup')
except:
    print('DB and Table set up were not successfull, please try again')

Successfully pulled from pneumonia_db


In [16]:
df_meta.head(2)

,image_id,raw_s3_key,preprocessed_s3_key,label,label_int,split,source,file_type,pixel_mean,pixel_std,img_height,img_width,event_time
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,raw-images/train/NORMAL/0004cfab-14fd-4e49-80b...,preprocessed-images/train/NORMAL/0004cfab-14fd...,NORMAL,0,train,rsna,dcm,129.9480,72.6177,512,512,2026-06-03T21:58:09Z
1,0022995a-45eb-4cfa-9a59-cd15f5196c64,raw-images/train/NORMAL/0022995a-45eb-4cfa-9a5...,preprocessed-images/train/NORMAL/0022995a-45eb...,NORMAL,0,train,rsna,dcm,122.4426,63.1398,512,512,2026-06-03T21:58:09Z
